# 1 多层感知机

多层感知机（Multilayer Perceptron, MLP）在单层神经网络的基础上引入一个或多个隐藏层。 隐藏层的输出通过激活函数进行非线性变换，使网络具备拟合复杂函数的能力。 常见的激活函数包括 ReLU、Sigmoid 和 Tanh。

In [1]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import numpy as np

In [2]:
x = torch.arange(-8.0, 8.0, 0.1, requires_grad=True)
y = torch.relu(x)
plt.plot(x.detach(), y.detach(), label='ReLU')
plt.plot(x.detach(), torch.sigmoid(x).detach(), label='Sigmoid')
plt.plot(x.detach(), torch.tanh(x).detach(), label='Tanh')
plt.legend()
plt.grid(True)
plt.show()

# 2 从零开始实现

我们使用 Fashion-MNIST 数据集，手动实现一个含单隐藏层的多层感知机。

In [3]:
batch_size = 256
transform = transforms.ToTensor()
mnist_train = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
mnist_test = torchvision.datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)
train_iter = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_iter = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

In [4]:
num_inputs, num_hiddens, num_outputs = 784, 256, 10
W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))
params = [W1, b1, W2, b2]

In [5]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

def net(X):
    X = X.reshape(-1, num_inputs)
    H = relu(X @ W1 + b1)
    return H @ W2 + b2

loss = nn.CrossEntropyLoss(reduction='none')

In [6]:
num_epochs, lr = 5, 0.5
optimizer = torch.optim.SGD(params, lr=lr)

def evaluate_accuracy(data_iter, net, loss_fn):
    acc_sum, n = 0.0, 0
    for X, y in data_iter:
        y_hat = net(X)
        acc_sum += (y_hat.argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    return acc_sum / n

for epoch in range(num_epochs):
    train_loss, train_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y).sum()
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        train_loss += l.item()
        train_acc += (y_hat.argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    test_acc = evaluate_accuracy(test_iter, net, loss)
    print(f'epoch {epoch+1}, loss {train_loss/n:.4f}, train acc {train_acc/n:.3f}, test acc {test_acc:.3f}')

epoch 1, loss 0.7923, train acc 0.650, test acc 0.673
epoch 2, loss 0.5722, train acc 0.786, test acc 0.776
epoch 3, loss 0.5057, train acc 0.814, test acc 0.800
epoch 4, loss 0.4746, train acc 0.828, test acc 0.810
epoch 5, loss 0.4545, train acc 0.837, test acc 0.818


# 3 简洁实现

使用 nn.Sequential 和 nn.Linear, nn.ReLU 快速搭建 MLP。

In [7]:
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights)

In [8]:
batch_size = 256
train_iter = DataLoader(mnist_train, batch_size, shuffle=True)
test_iter = DataLoader(mnist_test, batch_size, shuffle=False)

num_epochs, lr = 5, 0.1
optimizer = torch.optim.SGD(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    train_loss, train_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        train_loss += l.item() * y.shape[0]
        train_acc += (y_hat.argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    test_acc = evaluate_accuracy(test_iter, net, loss)
    print(f'epoch {epoch+1}, loss {train_loss/n:.4f}, train acc {train_acc/n:.3f}, test acc {test_acc:.3f}')

epoch 1, loss 0.7867, train acc 0.664, test acc 0.698
epoch 2, loss 0.5717, train acc 0.789, test acc 0.803
epoch 3, loss 0.5127, train acc 0.814, test acc 0.821
epoch 4, loss 0.4800, train acc 0.827, test acc 0.830
epoch 5, loss 0.4598, train acc 0.836, test acc 0.833


# 4 模型选择、欠拟合和过拟合

我们使用多项式回归来演示欠拟合和过拟合。通过控制数据生成的多项式阶数（真实函数）与模型的多项式阶数（拟合能力）来观察训练误差和测试误差的差异。

In [9]:
max_degree = 20
n_train, n_test = 100, 100
true_w = torch.zeros(max_degree)
true_w[0:4] = torch.tensor([5, 1.2, -3.4, 5.6])

features = torch.randn(n_train + n_test, 1)
poly_features = torch.cat([features ** i for i in range(max_degree)], dim=1)
labels = (poly_features @ true_w).reshape(-1) + torch.normal(0, 0.1, (n_train + n_test,))

In [10]:
def train_poly(degree):
    X_train = poly_features[:n_train, :degree]
    X_test = poly_features[n_train:, :degree]
    y_train = labels[:n_train]
    y_test = labels[n_train:]

    net = nn.Linear(degree, 1, bias=False)
    loss = nn.MSELoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

    for epoch in range(100):
        l = loss(net(X_train), y_train)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()

    train_loss = loss(net(X_train), y_train).item()
    test_loss = loss(net(X_test), y_test).item()
    print(f'degree={degree}: train loss {train_loss:.3f}, test loss {test_loss:.3f}')

for d in [1, 4, 20]:
    train_poly(d)

degree=1: train loss 9.492, test loss 8.651
degree=4: train loss 0.009, test loss 0.007
degree=20: train loss 0.004, test loss 6.961


In [11]:
degrees = range(1, 20)
train_losses, test_losses = [], []
for d in degrees:
    X_train = poly_features[:n_train, :d]
    X_test = poly_features[n_train:, :d]
    y_train = labels[:n_train]
    y_test = labels[n_train:]
    net_d = nn.Linear(d, 1, bias=False)
    loss_fn = nn.MSELoss()
    opt = torch.optim.SGD(net_d.parameters(), lr=0.01)
    for _ in range(100):
        opt.zero_grad()
        l = loss_fn(net_d(X_train), y_train)
        l.backward()
        opt.step()
    train_losses.append(loss_fn(net_d(X_train), y_train).detach().item())
    test_losses.append(loss_fn(net_d(X_test), y_test).detach().item())

plt.plot(degrees, train_losses, label='Train loss')
plt.plot(degrees, test_losses, label='Test loss')
plt.yscale('log')
plt.xlabel('Polynomial degree')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 5 权重衰减

权重衰减（L2 正则化）通过在损失函数中加入参数的 L2 范数惩罚项来限制模型复杂度，防止过拟合。

In [12]:
n_train, n_test = 20, 100
X = torch.randn(n_train + n_test, 200)
true_w = torch.ones(200) * 0.01
y = X @ true_w + torch.normal(0, 0.1, (n_train + n_test,))

X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

# 手动实现权重衰减
net1 = nn.Linear(200, 1, bias=False)
loss_fn = nn.MSELoss()
opt1 = torch.optim.SGD(net1.parameters(), lr=0.01)
lambd = 3.0

for epoch in range(50):
    l = loss_fn(net1(X_train), y_train) + lambd * net1.weight.pow(2).sum() / 2
    opt1.zero_grad()
    l.backward()
    opt1.step()

print(f'L2 norm of w (manual): {net1.weight.norm().item():.3f}')

# 使用优化器的 weight_decay
net2 = nn.Linear(200, 1, bias=False)
opt2 = torch.optim.SGD(net2.parameters(), lr=0.01, weight_decay=lambd)

for epoch in range(50):
    l = loss_fn(net2(X_train), y_train)
    opt2.zero_grad()
    l.backward()
    opt2.step()

print(f'L2 norm of w (optimizer): {net2.weight.norm().item():.3f}')

L2 norm of w (manual): 0.472
L2 norm of w (optimizer): 0.529


# 6 Dropout

Dropout 在训练过程中随机丢弃隐藏层神经元，相当于对神经网络进行正则化，减轻过拟合。

In [13]:
def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    if dropout == 1:
        return torch.zeros_like(X)
    if dropout == 0:
        return X
    mask = torch.rand(X.shape) > dropout
    return mask.float() * X / (1.0 - dropout)

In [14]:
X_demo = torch.arange(12, dtype=torch.float32).reshape(1, -1)
print(f'Original:\n{X_demo}')
print(f'Dropout p=0.5:\n{dropout_layer(X_demo, 0.5)}')
print(f'Dropout p=0.0:\n{dropout_layer(X_demo, 0.0)}')

In [15]:
batch_size = 256
train_iter = DataLoader(mnist_train, batch_size, shuffle=True)
test_iter = DataLoader(mnist_test, batch_size, shuffle=False)

num_inputs, num_hiddens1, num_hiddens2, num_outputs = 784, 256, 128, 10
dropout1, dropout2 = 0.2, 0.5

class DropoutMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens1) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(num_hiddens1))
        self.W2 = nn.Parameter(torch.randn(num_hiddens1, num_hiddens2) * 0.01)
        self.b2 = nn.Parameter(torch.zeros(num_hiddens2))
        self.W3 = nn.Parameter(torch.randn(num_hiddens2, num_outputs) * 0.01)
        self.b3 = nn.Parameter(torch.zeros(num_outputs))

    def forward(self, X):
        X = X.reshape(-1, num_inputs)
        H1 = relu(X @ self.W1 + self.b1)
        if self.training:
            H1 = dropout_layer(H1, dropout1)
        H2 = relu(H1 @ self.W2 + self.b2)
        if self.training:
            H2 = dropout_layer(H2, dropout2)
        return H2 @ self.W3 + self.b3

net_dropout = DropoutMLP()
loss = nn.CrossEntropyLoss(reduction='none')
optimizer = torch.optim.SGD(net_dropout.parameters(), lr=0.5)

for epoch in range(10):
    train_loss, train_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net_dropout(X)
        l = loss(y_hat, y).sum()
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        train_loss += l.item()
        train_acc += (y_hat.argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    test_acc = evaluate_accuracy(test_iter, net_dropout, loss)
    print(f'epoch {epoch+1}, train loss {train_loss/n:.4f}, test acc {test_acc:.3f}')

epoch 1, train loss 0.7167, test acc 0.702
epoch 2, train loss 0.5926, test acc 0.743
epoch 3, train loss 0.5686, test acc 0.714
epoch 4, train loss 0.5592, test acc 0.722
epoch 5, train loss 0.5537, test acc 0.756
epoch 6, train loss 0.5499, test acc 0.746
epoch 7, train loss 0.5469, test acc 0.753
epoch 8, train loss 0.5450, test acc 0.763
epoch 9, train loss 0.5431, test acc 0.760
epoch 10, train loss 0.5417, test acc 0.750


In [16]:
# 使用 nn.Dropout 简洁实现
net_dropout_simple = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(dropout1),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(dropout2),
    nn.Linear(128, 10)
)

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net_dropout_simple.apply(init_weights)

In [17]:
optimizer = torch.optim.SGD(net_dropout_simple.parameters(), lr=0.5)
loss = nn.CrossEntropyLoss()

for epoch in range(10):
    train_loss, train_acc, n = 0.0, 0.0, 0
    for X, y in train_iter:
        y_hat = net_dropout_simple(X)
        l = loss(y_hat, y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
        train_loss += l.item() * y.shape[0]
        train_acc += (y_hat.argmax(dim=1) == y).float().sum().item()
        n += y.shape[0]
    test_acc = evaluate_accuracy(test_iter, net_dropout_simple, loss)
    print(f'epoch {epoch+1}, train loss {train_loss/n:.4f}, test acc {test_acc:.3f}')

epoch 1, train loss 0.7288, test acc 0.731
epoch 2, train loss 0.5882, test acc 0.778
epoch 3, train loss 0.5545, test acc 0.796
epoch 4, train loss 0.5386, test acc 0.797
epoch 5, train loss 0.5276, test acc 0.810
epoch 6, train loss 0.5202, test acc 0.813
epoch 7, train loss 0.5151, test acc 0.812
epoch 8, train loss 0.5106, test acc 0.808
epoch 9, train loss 0.5070, test acc 0.810
epoch 10, train loss 0.5039, test acc 0.807
